
# N2: MVN-aware Comparison Notebook

This notebook extends the previous comparison by incorporating **multivariate normal (MVN) joint ranking probabilities** for the 4-asset case. It compares four approaches:

1. **Presentation (product-of-probabilities)**: pair spreads vs base (Kirk/Margrabe) and scale by pairwise outranking probabilities.
2. **LZD-ext (pairwise conditional)**: replace Kirk with Li-Zhou-Deng 1-D conditional quadrature for the pair pricer; keep the same ranking scaffold.
3. **Presentation-MVN (joint ranking)**: same as (1) but, for the 4th leg, use the **joint** probability P(S4>S2, S4>S3) computed from a **bivariate normal** over log-differences.
4. **Monte Carlo (benchmark)**: correlated GBMs with antithetic variates.

We keep K=0 by default so pair pricing is Margrabe-exact for two legs.


## 1. Imports & Utilities

In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.hermite import hermgauss
from time import perf_counter
from dataclasses import dataclass
from typing import Tuple

from scipy.stats import norm

plt.style.use('seaborn-v0_8')
np.random.seed(42)


## 2. Core Building Blocks

In [2]:

# Black-76 style helper on forward

def black_call(F, K, vol_sqrtT):
    F = float(F); K = float(K)
    if vol_sqrtT <= 1e-12:
        return max(F - K, 0.0)
    d1 = (np.log(F / K) + 0.5 * vol_sqrtT**2) / vol_sqrtT
    d2 = d1 - vol_sqrtT
    return F * norm.cdf(d1) - K * norm.cdf(d2)

# Margrabe spread (exact when K=0)

def margrabe(F1, F2, sigma1, sigma2, rho, T):
    v = (sigma1**2 + sigma2**2 - 2*rho*sigma1*sigma2) * T
    vs = np.sqrt(max(v, 0.0))
    if vs <= 1e-14:
        return max(F2 - F1, 0.0)
    d1 = (np.log(F2/F1) + 0.5*v) / vs
    d2 = d1 - vs
    return F2 * norm.cdf(d1) - F1 * norm.cdf(d2)

# Kirk approximation (general K) with Margrabe fallback

def kirk(F1, F2, sigma1, sigma2, rho, T, K=0.0):
    if abs(K) < 1e-15:
        return margrabe(F1, F2, sigma1, sigma2, rho, T)
    beta = F2 / (F2 + K)
    v = (sigma1**2 - 2*beta*rho*sigma1*sigma2 + (beta**2)*sigma2**2) * T
    vs = np.sqrt(max(v, 0.0))
    d1 = (np.log(F1/(F2+K)) + 0.5*v) / vs
    d2 = d1 - vs
    return F1 * norm.cdf(d1) - (F2 + K) * norm.cdf(d2)

# LZD: conditional quadrature for pair spread

def lzd_conditional(F1, F2, sigma1, sigma2, rho, T, K=0.0, n=32):
    z, w = hermgauss(n)
    mu1 = np.log(F1) - 0.5 * (sigma1**2) * T
    s1 = sigma1 * np.sqrt(T)
    v2_cond = (1 - rho**2) * sigma2**2 * T
    total = 0.0
    for zi, wi in zip(z, w):
        lnS1 = mu1 + np.sqrt(2.0) * s1 * zi
        S1 = np.exp(lnS1)
        mu2_given1 = (np.log(F2) - 0.5*sigma2**2*T) + (rho * sigma2 / sigma1) * (lnS1 - mu1)
        F2_cond = np.exp(mu2_given1 + 0.5 * v2_cond)
        K_eff = S1 + K
        if K_eff <= 1e-15:
            price_cond = F2_cond
        else:
            price_cond = black_call(F2_cond, K_eff, np.sqrt(v2_cond))
        total += wi * price_cond
    return float(total / np.sqrt(np.pi))

# Simple outranking probability for lognormals via logs (univariate)

def outrank_prob(Fi, Fj, si, sj, rho_ij, T):
    mu = (np.log(Fi) - 0.5*si**2*T) - (np.log(Fj) - 0.5*sj**2*T)
    var = (si**2 + sj**2 - 2*rho_ij*si*sj) * T
    s = np.sqrt(max(var, 1e-18))
    return float(norm.cdf(mu / s))

@dataclass
class Market:
    F: np.ndarray
    sigma: np.ndarray
    rho: np.ndarray
    T: float = 1.0
    K: float = 0.0
    r: float = 0.0
    def check(self):
        m = len(self.F)
        assert self.sigma.shape == (m,), 'sigma shape mismatch'
        assert self.rho.shape == (m,m), 'rho shape mismatch'
        ev = np.linalg.eigvalsh(self.rho)
        if ev.min() < -1e-6:
            raise ValueError('Correlation matrix not PSD')


## 3. MVN Joint Ranking Probability for 4-Asset Case

In [7]:

# We want P(S4>S2, S4>S3).
# Let X = ln S be multivariate normal with mean mu_i and covariance C_ij.
# Define Y1 = X4 - X2, Y2 = X4 - X3. Then [Y1, Y2] is bivariate normal with:
#   mean: [mu4-mu2, mu4-mu3]
#   cov: [[Var4+Var2-2Cov42,  Var4 - Cov43 - Cov42 + Cov23],
#         [sym,               Var4+Var3-2Cov43            ]]
# We compute P(Y1>0, Y2>0) using SciPy's mvn.mvnun if available; fallback to MC.

from scipy.stats import multivariate_normal


def joint_prob_S4_beats_S2_S3(F, sigma, R, T):
    # inputs are vectors for 4 assets (index 0..3). We use assets 1..3 as base/alternatives,
    # but here we only need indices 1(S2), 2(S3), 3(S4).
    # Build mu, cov for X=ln S
    sig = np.array(sigma)
    var = (sig**2) * T
    mu = np.log(F) - 0.5 * var
    C = (sig[:,None] * sig[None,:] * R) * T
    # indices for 2,3,4 in 1-based terms are 1,2,3 in 0-based
    i2, i3, i4 = 1, 2, 3
    # mean vector for Y
    m1 = mu[i4] - mu[i2]
    m2 = mu[i4] - mu[i3]
    # cov elements
    v4 = var[i4]; v2 = var[i2]; v3 = var[i3]
    c42 = C[i4, i2]; c43 = C[i4, i3]; c23 = C[i2, i3]
    s11 = v4 + v2 - 2.0*c42
    s22 = v4 + v3 - 2.0*c43
    s12 = v4 - c43 - c42 + c23
    Sigma = np.array([[s11, s12],[s12, s22]], dtype=float)
    muY = np.array([m1, m2], dtype=float)

    # P(Y1>0, Y2>0) = 1 - P(Y1<=0) - P(Y2<=0) + P(Y1<=0,Y2<=0)
    p_y1_le0 = norm.cdf(-muY[0] / np.sqrt(max(Sigma[0, 0], 0.0)))
    p_y2_le0 = norm.cdf(-muY[1] / np.sqrt(max(Sigma[1, 1], 0.0)))

    try:
        c00 = multivariate_normal(mean=muY, cov=Sigma).cdf([0.0, 0.0])
        return float(1.0 - p_y1_le0 - p_y2_le0 + c00)
    except Exception:
        # Fallback: Monte Carlo on the bivariate normal Y
        rng = np.random.default_rng(12345)
        Z = rng.multivariate_normal(mean=muY, cov=Sigma, size=400000)
        return float(np.mean((Z[:,0]>0) & (Z[:,1]>0)))


## 4. Models: Presentation, LZD-ext, MVN-aware (4-asset), and Monte Carlo

In [4]:

# Presentation (product-of-probabilities)

def presentation_price(mkt: Market):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    base = 0
    alts = np.argsort(-F[1:]) + 1
    terms = []
    def spread(i):
        return kirk(F[base], F[i], s[base], s[i], R[base, i], T, K)
    price = 0.0
    if len(alts) >= 1:
        p2 = spread(alts[0]); price += p2; terms.append((int(alts[0]+1), p2, 1.0))
    if len(alts) >= 2:
        i3 = alts[1]; p3 = spread(i3)
        P32 = outrank_prob(F[i3], F[alts[0]], s[i3], s[alts[0]], R[i3, alts[0]], T)
        price += p3 * P32; terms.append((int(i3+1), p3, P32))
    if len(alts) >= 3:
        i4 = alts[2]; p4 = spread(i4)
        P42 = outrank_prob(F[i4], F[alts[0]], s[i4], s[alts[0]], R[i4, alts[0]], T)
        P43 = outrank_prob(F[i4], F[alts[1]], s[i4], s[alts[1]], R[i4, alts[1]], T)
        price += p4 * P42 * P43; terms.append((int(i4+1), p4, P42*P43))
    return price, terms

# LZD-ext (pairwise conditional) with same ranking scaffold

def lzd_ext_price(mkt: Market, n_gh: int = 32):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    base = 0
    alts = np.argsort(-F[1:]) + 1
    def spread_lzd(i):
        return lzd_conditional(F[base], F[i], s[base], s[i], R[base, i], T, K, n=n_gh)
    price = 0.0; terms = []
    if len(alts) >= 1:
        p2 = spread_lzd(alts[0]); price += p2; terms.append((int(alts[0]+1), p2, 1.0))
    if len(alts) >= 2:
        i3 = alts[1]; p3 = spread_lzd(i3)
        P32 = outrank_prob(F[i3], F[alts[0]], s[i3], s[alts[0]], R[i3, alts[0]], T)
        price += p3 * P32; terms.append((int(i3+1), p3, P32))
    if len(alts) >= 3:
        i4 = alts[2]; p4 = spread_lzd(i4)
        P42 = outrank_prob(F[i4], F[alts[0]], s[i4], s[alts[0]], R[i4, alts[0]], T)
        P43 = outrank_prob(F[i4], F[alts[1]], s[i4], s[alts[1]], R[i4, alts[1]], T)
        price += p4 * P42 * P43; terms.append((int(i4+1), p4, P42*P43))
    return price, terms

# Presentation-MVN: replace last term weight by true joint MVN probability

def presentation_mvn_price(mkt: Market):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    base = 0
    alts = np.argsort(-F[1:]) + 1
    def spread(i):
        return kirk(F[base], F[i], s[base], s[i], R[base, i], T, K)
    price = 0.0
    terms = []
    if len(alts) >= 1:
        p2 = spread(alts[0]); price += p2; terms.append((int(alts[0]+1), p2, 1.0))
    if len(alts) >= 2:
        i3 = alts[1]; p3 = spread(i3)
        P32 = outrank_prob(F[i3], F[alts[0]], s[i3], s[alts[0]], R[i3, alts[0]], T)
        price += p3 * P32; terms.append((int(i3+1), p3, P32))
    if len(alts) >= 3:
        # Use MVN joint rank prob for i4 outranking both i2 and i3
        i2 = alts[0]; i4 = alts[2]
        p4 = spread(i4)
        # Build a 4-asset vector [base,i2,i3,i4] to feed the joint function
        # We need ordering so that indices 1,2,3 correspond to S2,S3,S4 as in joint_prob function
        F4 = np.array([F[base], F[i2], F[alts[1]], F[i4]], dtype=float)
        s4 = np.array([s[base], s[i2], s[alts[1]], s[i4]], dtype=float)
        R4 = R[[base, i2, alts[1], i4]][:, [base, i2, alts[1], i4]].astype(float)
        P4_joint = joint_prob_S4_beats_S2_S3(F4, s4, R4, T)
        price += p4 * P4_joint; terms.append((int(i4+1), p4, P4_joint))
    return price, terms

# Monte Carlo benchmark

def mc_price(mkt: Market, n_sims: int = 100000, antithetic: bool = True, seed: int = 7):
    mkt.check()
    F, s, R, T, K = mkt.F, mkt.sigma, mkt.rho, mkt.T, mkt.K
    rng = np.random.default_rng(seed)
    m = len(F)
    L = np.linalg.cholesky(R + 1e-12*np.eye(m))
    n = n_sims
    if antithetic:
        n = (n_sims + 1)//2
    Z = rng.standard_normal((n, m))
    if antithetic:
        Z = np.vstack([Z, -Z])
    Z = Z[:n_sims]
    shocks = Z @ L.T
    drift = (-0.5 * (s**2) * T)
    vol = s * np.sqrt(T)
    lnS = np.log(F) + drift + shocks * vol
    S = np.exp(lnS)
    baseS = S[:, 0]
    alts = S[:, 1:]
    payoff = np.maximum(alts - baseS[:, None] - K, 0.0)
    best = payoff.max(axis=1)
    return best.mean()


## 5. Test Scenarios and Experiment Runner

In [8]:

# Example parameters (similar to previous, but ensure PSD)
F3 = (50.0, 52.0, 51.0)
sig3 = (0.35, 0.30, 0.30)
# Use uniform corr for PSD sweepability
R3 = np.array([[1.0, 0.6, 0.6],
               [0.6, 1.0, 0.6],
               [0.6, 0.6, 1.0]])

F4 = (50.0, 52.0, 51.0, 49.5)
sig4 = (0.35, 0.30, 0.30, 0.32)
R4 = np.array([[1.0, 0.6, 0.5, 0.55],
               [0.6, 1.0, 0.55, 0.5],
               [0.5, 0.55, 1.0, 0.6],
               [0.55, 0.5, 0.6, 1.0]])

T = 0.5
K = 0.0

rows = []

# 3-asset case: MVN joint is identical to pairwise outrank (only one competitor), so focus on first two methods vs MC
mkt3 = Market(F=np.array(F3,dtype=float), sigma=np.array(sig3,dtype=float), rho=R3.astype(float), T=T, K=K)

# 4-asset case: compare presentation vs presentation-MVN vs LZD-ext vs MC
mkt4 = Market(F=np.array(F4,dtype=float), sigma=np.array(sig4,dtype=float), rho=R4.astype(float), T=T, K=K)

# 3-asset
t0=perf_counter(); v_pres3,_ = presentation_price(mkt3); t1=perf_counter()
t2=perf_counter(); v_lzd3,_  = lzd_ext_price(mkt3, n_gh=32); t3=perf_counter()
t4=perf_counter(); v_mc3     = mc_price(mkt3, n_sims=200000, antithetic=True, seed=11); t5=perf_counter()

rows.append({
    'Case':'3-assets','Presentation':v_pres3,'LZD-ext':v_lzd3,'Pres-MVN':np.nan,'MonteCarlo':v_mc3,
    't_pres_ms':(t1-t0)*1e3,'t_lzd_ms':(t3-t2)*1e3,'t_presmvn_ms':np.nan,'t_mc_ms':(t5-t4)*1e3
})

# 4-asset
u0=perf_counter(); v_pres4,_  = presentation_price(mkt4); u1=perf_counter()
u2=perf_counter(); v_lzd4,_   = lzd_ext_price(mkt4, n_gh=32); u3=perf_counter()
u4=perf_counter(); v_pmvn4,_  = presentation_mvn_price(mkt4); u5=perf_counter()
u6=perf_counter(); v_mc4      = mc_price(mkt4, n_sims=300000, antithetic=True, seed=21); u7=perf_counter()

rows.append({
    'Case':'4-assets','Presentation':v_pres4,'LZD-ext':v_lzd4,'Pres-MVN':v_pmvn4,'MonteCarlo':v_mc4,
    't_pres_ms':(u1-u0)*1e3,'t_lzd_ms':(u3-u2)*1e3,'t_presmvn_ms':(u5-u4)*1e3,'t_mc_ms':(u7-u6)*1e3
})

res = pd.DataFrame(rows)
res


,Case,Presentation,LZD-ext,Pres-MVN,MonteCarlo,t_pres_ms,t_lzd_ms,t_presmvn_ms,t_mc_ms
0,3-assets,7.457258,7.457258,NaN,7.321075,1.3566,27.3826,NaN,32.7969
1,4-assets,8.430091,8.430091,8.797673,8.846460,1.1262,25.7227,2.0471,60.0410


## 6. Plots: Values and Runtimes

In [ ]:

for case in res['Case']:
    row = res[res['Case']==case].iloc[0]
    labs = ['Presentation','LZD-ext'] if case=='3-assets' else ['Presentation','LZD-ext','Pres-MVN','MonteCarlo']
    vals = [row[l] for l in labs]
    times= [row['t_pres_ms'], row['t_lzd_ms']] if case=='3-assets' else [row['t_pres_ms'],row['t_lzd_ms'],row['t_presmvn_ms'],row['t_mc_ms']]

    fig, axes = plt.subplots(1,2, figsize=(11,4))
    axes[0].bar(labs, vals, color=['#2a9d8f','#264653','#8ab17d','#e76f51'][:len(labs)])
    axes[0].set_title(f"{case}: Values"); axes[0].set_ylabel('Option Value'); axes[0].grid(True, axis='y', alpha=0.3)

    axes[1].bar(labs, times, color=['#2a9d8f','#264653','#8ab17d','#e76f51'][:len(labs)])
    axes[1].set_title(f"{case}: Runtime (ms)"); axes[1].set_ylabel('Milliseconds'); axes[1].grid(True, axis='y', alpha=0.3)

    plt.tight_layout(); plt.show()


## 7. 4-Asset Applicability Stress: Absolute Error vs Correlation (uniform PSD grid)

In [ ]:

# Sweep a uniform correlation c for PSD, and compare absolute error vs MC for 4-asset methods
corr_levels = np.linspace(0.0, 0.9, 10)
err_pres, err_lzd, err_pmvn = [], [], []

for c in corr_levels:
    R = np.array([[1.0, c,   c,   c  ],
                  [c,   1.0, c,   c  ],
                  [c,   c,   1.0, c  ],
                  [c,   c,   c,   1.0]])
    mk = Market(F=np.array(F4), sigma=np.array(sig4), rho=R, T=T, K=K)
    vp,_ = presentation_price(mk)
    vl,_ = lzd_ext_price(mk)
    vm,_ = presentation_mvn_price(mk)
    vM   = mc_price(mk, n_sims=200000, antithetic=True, seed=777)
    err_pres.append(abs(vp - vM))
    err_lzd.append(abs(vl - vM))
    err_pmvn.append(abs(vm - vM))

plt.figure(figsize=(7,4))
plt.plot(corr_levels, err_pres, label='Presentation abs error')
plt.plot(corr_levels, err_lzd, label='LZD-ext abs error')
plt.plot(corr_levels, err_pmvn, label='Pres-MVN abs error', linewidth=2)
plt.xlabel('Uniform correlation c'); plt.ylabel('Absolute error vs MC')
plt.title('4-asset: Error vs correlation (uniform PSD)')
plt.grid(True, alpha=0.3); plt.legend(); plt.show()


## 8. Summary Table

In [ ]:

view = res.copy()
for col in ['Presentation','LZD-ext','Pres-MVN','MonteCarlo']:
    if col in view:
        view[col] = view[col].astype(float)
if 'Pres-MVN' not in view.columns:
    view['Pres-MVN'] = np.nan
view['Pres_vs_MC_%']   = 100.0*(view['Presentation']/view['MonteCarlo'] - 1)
view['LZD_vs_MC_%']    = 100.0*(view['LZD-ext']/view['MonteCarlo'] - 1)
view['PMVN_vs_MC_%']   = 100.0*(view['Pres-MVN']/view['MonteCarlo'] - 1)
view.round(5)
